## Решение задачи A от Кулибабы Степана


### Основная стратегия — блендинг моделей

Я заметил, что разные модели автоматического распознавания речи (ASR) имеют разную точность, но при этом практически не дают ложноположительных результатов (когда модель ошибочно определяет проблему, присваивая 1 вместо 0). Этот вывод натолкнул меня на идею **ансамбля (блендинга)** нескольких моделей.

Моя финальная стратегия заключается в использовании логического **ИЛИ (OR)** для предсказаний: если **хотя бы одна** из моделей транскрибирует фразу, сигнализирующую о проблеме (такие как «не слышу», «не слышно» или их возможные опечатки), я устанавливаю окончательную метку в **1**.

### Модели, которые я использовал

Для транскрипции аудио я задействовал три различные ASR-модели:

1.  **Whisper (Модель `large`):**
    * Я загружаю модель `large` и использую ее с явным указанием русского языка (`ru`).
    * Я присваиваю метку **1**, если в транскрипции обнаруживаю `"не слышу"`, `"не слышно"`, `"неслышу"` или `"неслышо"` (в нижнем регистре).

2.  **GigaAM-v2-RNNT:**
    * Я клонирую репозиторий GigaAM и использую модель `v2_rnnt`.
    * Я присваиваю метку **1**, если транскрипция содержит `"не слышу"`, `"неслышу"`, `"не слышно"` или `"неслышно"` (в нижнем регистре).

3.  **Gemini 2.5 Pro:**
    * Я использую облачный API `gemini-2.5-pro` для транскрипции, загружая и удаляя аудиофайлы через клиент.
    * Для обнаружения проблемы я ищу `"не слышу"`, `"неслышу"`, `"не слышно"` или `"неслышно"` (в нижнем регистре).

### Объединение Результатов

В конце я считываю предсказания (`label`) от каждой модели (`df_whisper`, `df_giga`, `df_gemini`) и объединяю их с помощью оператора логического ИЛИ (`|`) для получения финальной метки в файле `submission.csv`:

```python
submit['label'] = df_whisper['label'] | df_giga['label'] | df_gemini['label']
```

## Важные моменты для воспроизведения моего решения

1.  **Требования к Железу:** Для запуска ASR-моделей Whisper и GigaAM необходима **видеокарта (GPU)**.

2.  **Пути к Данным:** Поскольку я разрабатывал решение в среде Kaggle, все пути к данным, такие как путь к файлу `sample_submission_1.csv`, являются абсолютными: `/kaggle/input/task-1-vseross/sample_submission_1.csv`. При запуске в другой среде (например, на локальной машине) нужно будет заменить этот путь на **свой**.

3.  **API-ключ для Gemini:** Для работы с моделью Gemini 2.5 Pro  требуется действующий API-ключ. Его необходимо вставить в соответствующей ячейке кода:
    * **Необходимо заменить:** `'YOUR_API_KEY'` на **свой API-ключ**.

#### Скачиваем данные:

In [33]:
import os
import re
import requests
import subprocess
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm

def get_direct_file_link(mailru_file_url: str) -> str:
    """
    Преобразует публичную ссылку вида:
        https://cloud.mail.ru/public/<key>/<subkey>/<filename>
    в прямую ссылку на CDN, по которой можно скачать файл через wget или requests.

    Возвращает прямую ссылку для скачивания.
    """
    resp = requests.get(mailru_file_url)
    if resp.status_code != 200:
        raise RuntimeError(f"Ошибка {resp.status_code} при запросе {mailru_file_url}")

    match = re.search(r'dispatcher.*?weblink_get.*?url":"(.*?)"', resp.text)
    if not match:
        raise RuntimeError("Не удалось найти CDN ссылку в HTML Mail.ru")

    base_url = match.group(1)
    parts = mailru_file_url.strip("/").split("/")[-3:]
    return f"{base_url}/{parts[0]}/{parts[1]}/{parts[2]}"


def download_from_mailru(file_url: str, local_name: str, force: bool = False, show_progress: bool = True):
    """
    Скачивает файл с Mail.ru по публичной ссылке.

    Args:
        file_url: ссылка на файл в облаке Mail.ru.
        local_name: имя файла для сохранения.
        force: если True — перекачивает даже если файл уже есть.
        show_progress: показывать ли прогресс-бар.
    """
    local_path = Path(local_name)
    if local_path.exists() and not force:
        print(f"Файл {local_name} уже существует, пропускаем скачивание.")
        return

    direct = get_direct_file_link(file_url)
    print(f"Скачиваем {file_url} → {local_name}")

    with requests.get(direct, stream=True) as r:
        r.raise_for_status()
        total_size = int(r.headers.get("content-length", 0))
        block_size = 8192
        with open(local_name, "wb") as f, tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=f"Downloading {local_name}",
            disable=not show_progress,
        ) as bar:
            for chunk in r.iter_content(block_size):
                f.write(chunk)
                bar.update(len(chunk))

    print(f"Файл {local_name} успешно скачан ({os.path.getsize(local_name)/1e6:.1f} MB).")

In [2]:
test_link  = "https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/test_data.tar"

In [ ]:
download_from_mailru(test_link, "test_data.tar")

In [ ]:
subprocess.run(["tar", "xf", "test_data.tar"],
               check=True,
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

## Часть 1 - предсказания от модели whisper

In [ ]:
!pip install -qq -U openai-whisper

In [7]:
import whisper

def transcribe_opus_ids(id_list, folder=".", model_name="small", language=None):
    """
    Транскрибирует файлы вида ._{id}.opus и возвращает список текстов.

    Args:
        id_list (list[str]): список ID, например ['_12312312', '_98765432']
        folder (str | Path): путь к папке с .opus файлами
        model_name (str): название модели whisper ('tiny', 'base', 'small', 'medium', 'large')
        language (str | None): язык ('ru', 'en', ...). Если None — автоопределение

    Returns:
        list[str]: список текстов (по порядку id_list)
    """
    folder = Path(folder)
    model = whisper.load_model(model_name).to('cuda')

    texts = []
    for file_id in tqdm(id_list):
        filename = f"{file_id}.opus"
        file_path = folder / filename

        if not file_path.exists():
            print(f"Файл не найден: {file_path}")
            texts.append("")
            continue

        result = model.transcribe(str(file_path), language=language)
        texts.append(result.get("text", "").strip())

    return texts

In [ ]:
submit = pd.read_csv("/kaggle/input/task-1-vseross/sample_submission_1.csv")
ids = list(submit['id'])
texts = transcribe_opus_ids(ids, folder="test_opus/audio", model_name="large", language='ru')
labels = [[0, 1][int('не слышу' in str(text).lower().strip() or "не слышно" in str(text).lower().strip() or "неслышу" in str(text).lower().strip() or "неслышо" in str(text).lower().strip())] for text in texts]

submit['texts'] = texts
submit['label'] = labels
submit.to_csv("submission_whisper.csv", index=False)

## Часть 2 - предсказания от модели GigaAM-v2-RNNT

In [ ]:
!git clone https://github.com/salute-developers/GigaAM.git
%cd GigaAM
!pip install -e .
%cd ..

In [ ]:
import torch
import gigaam
import pandas as pd
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path


def transcribe_opus_ids(id_list, folder=".", model_name="v2_rnnt"):
    """
    Транскрибирует .opus файлы по списку id и возвращает список текстов.

    Args:
        id_list (list[str]): список ID, например ['_12312312', '_98765432']
        folder (str | Path): путь к папке с .opus файлами
        model_name (str): название модели GigaAM ('v2_rnnt', 'v2_conformer', ...)

    Returns:
        list[str]: список транскрибированных текстов (по порядку id_list)
    """
    folder = Path(folder)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Загружается модель {model_name} на {device}...")
    model = gigaam.load_model(model_name).to(device)

    texts = []
    for file_id in tqdm(id_list):
        file_path = folder / f"{file_id}.opus"

        if not file_path.exists():
            print(f"Файл не найден: {file_path}")
            texts.append("")
            continue

        try:
            text = model.transcribe(str(file_path)) or ""
        except Exception as e:
            print(f"[WARN] Ошибка при обработке {file_path}: {e}")
            text = ""

        texts.append(text.strip())

    return texts



submit = pd.read_csv("/kaggle/input/task-1-vseross/sample_submission_1.csv")
ids = list(submit["id"])
texts = transcribe_opus_ids(ids, folder="test_opus/audio", model_name="v2_rnnt")
labels = [
    int(
        "не слышу" in str(text).lower().strip()
        or "неслышу" in str(text).lower().strip()
        or "не слышно" in str(text).lower().strip()
        or "неслышно" in str(text).lower().strip()
    )
    for text in texts
]

submit["text"] = texts
submit["label"] = labels
submit.to_csv("submission_gigaam.csv", index=False)

# Часть 3 - предсказания от модели gemini

In [ ]:
!pip install google-genai

In [ ]:
import os
from google import genai
from google.genai import types
from tqdm import tqdm

def transcribe_audio(file_path: str):
    audio_file = client.files.upload(file=file_path)
    try:
        prompt = "Сгенерируйте полную транскрипцию речи из этого аудиофайла."
        response = client.models.generate_content(
            model='gemini-2.5-pro',
            contents=[prompt, audio_file]
        )
        text = response.text
    except Exception as err:
        text = 'no'
    finally:
        client.files.delete(name=audio_file.name)
    return text


submit = pd.read_csv("/kaggle/input/task-1-vseross/sample_submission_1.csv")
ids = list(submit["id"])

texts = []
client = genai.Client(api_key='YOUR_API_KEY')
for i in tqdm(ids):
    text = transcribe_audio(f'test_opus/audio/{i}.opus')
    texts.append(text)


labels = [
    int(
        "не слышу" in str(text).lower().strip()
        or "неслышу" in str(text).lower().strip()
        or "не слышно" in str(text).lower().strip()
        or "неслышно" in str(text).lower().strip()
    )
    for text in texts
]

submit["text"] = texts
submit["label"] = labels
submit.to_csv("submission_gemini.csv", index=False)

# Совмещение предсказаний

In [ ]:
df_whisper = pd.read_csv("submission_whisper.csv")
df_giga = pd.read_csv("submission_gigaam.csv")
df_gemini = pd.read_csv("submission_gemini.csv")
submit = pd.read_csv("/kaggle/input/task-1-vseross/sample_submission_1.csv")

submit['label'] = df_whisper['label'] | df_giga['label'] | df_gemini['label']
submit.to_csv("submission.csv")